# 3 · Make a reproducible RustGIS handoff

**Question:** what should accompany a GeoJSON file when it leaves a notebook?

The handoff package will contain selected features, a source note, a bounding box, and a count. RustGIS can open the GeoJSON; the note helps a reader avoid treating invented heights as measured values.

![Handoff diagram](https://jltobias.github.io/JupyterLite-RustGIS/assets/stac-rustgis-flow.svg)

## Select a focused layer

In [ ]:
import json
from pathlib import Path
source = json.loads(Path('data/demo_features.geojson').read_text())
polygons = [f for f in source['features'] if f['geometry']['type'] == 'Polygon']
print('Selected polygons:', len(polygons))

## Preserve the data structure

A filtered GeoJSON remains a FeatureCollection. Keep the original properties, including `height_m`, so you can inspect them in RustGIS.

In [ ]:
selection = {'type':'FeatureCollection', 'name':'Illustrative study footprints', 'features':polygons}
Path('selected_footprints.geojson').write_text(json.dumps(selection, indent=2), encoding='utf-8')
print('Saved selected_footprints.geojson')

## Write a small manifest

In [ ]:
manifest = {
    'source': 'JupyterLite-RustGIS/data/demo_features.geojson',
    'purpose': 'illustrative teaching fixture',
    'features': len(polygons),
    'fields': sorted({key for f in polygons for key in f['properties']}),
    'warning': 'invented heights; not surveyed data'
}
Path('selected_footprints_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print(json.dumps(manifest, indent=2))

## Check both files before download

In [ ]:
assert json.loads(Path('selected_footprints.geojson').read_text())['features'] == polygons
assert json.loads(Path('selected_footprints_manifest.json').read_text())['features'] == len(polygons)
print('Export and manifest agree.')

## Open it in RustGIS

Download both files from the JupyterLite file browser. In [RustGIS Web](https://web.rustgis.app/), choose **Add Data** and select the GeoJSON. Inspect the attribute table and symbolize by `height_m`. The manifest is a human-readable provenance note, not a RustGIS project file.

You can also test the pre-hosted, complete [teaching layer in RustGIS](https://web.rustgis.app/?blank&data=https://jltobias.github.io/JupyterLite-RustGIS/data/demo_features.geojson). A newly created browser file is **not automatically public at a URL**, so use the file picker for your own export.

## Native follow-through

The separate [native RustGIS notebook](https://github.com/jltobias/JupyterLite-RustGIS/blob/main/notebooks/04_native_rustgis.ipynb) uses `rustgis.Session(sample=True)`, map export, and a buffer command. It must run in local Jupyter because the native package is unavailable in Pyodide.

**Challenge:** filter the layer by a threshold such as `height_m >= 60`, update the manifest's feature count, and compare the selection with the 3D map.

**Sources:** [RustGIS](https://github.com/opengeos/RustGIS), [RustGIS web URL parameters](https://github.com/opengeos/RustGIS/blob/main/docs/installation.md#web-app), [your STAC Browser](https://jltobias.github.io/JupyterLite-STAC-Browser/explorer/), [STAC specification](https://stacspec.org/). All hosted data here is public teaching material. Check the provider's license before reusing STAC assets.